# Simulation Plotting Tools

## Import Data

In [1]:
import numpy as np
import ROOT
import pandas as pd
import random

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [3]:
class Particle:
    def __init__(self, track_id, init_pos, init_mom, actual_hits, kalman_hits):
        self.track_id = track_id
        self.init_pos = init_pos
        self.init_mom = init_mom
        self.actual_hits = actual_hits
        self.kalman_hits = kalman_hits


class LayerHit:
    def __init__(self, hit_pos, hit_mom):
        self.hit_pos = hit_pos
        self.hit_mom = hit_mom

In [7]:
filepath = "output/"

# Get the actual and kalman fitted data from the ROOT file
ROOT.gSystem.Load("output/libDriftChamberlib.so")
with ROOT.TFile.Open("output/kalman_output.root", "READ") as file:
    if not file or file.IsZombie():
        print("Error opening file")

    tree = file.Get("Particles")

    particles = []

    for entry in tree:
        for particle in entry.Particles:
            track_id = particle.getID()
            init_pos = (particle.getPosition().X(), particle.getPosition().Y(), particle.getPosition().Z())
            init_mom = (particle.getMomentum().X(), particle.getMomentum().Y(), particle.getMomentum().Z())
            
            actual_hits = []
            kalman_hits = []

            for hit in particle.getActualHits():
                pos = (hit.getEntryPosition().X(), hit.getEntryPosition().Y(), hit.getEntryPosition().Z())
                mom = (hit.getEntryMomentum().X(), hit.getEntryMomentum().Y(), hit.getEntryMomentum().Z())

                layer_hit = LayerHit(pos, mom)
                actual_hits.append(layer_hit)
            
            for hit in particle.getKalmanHits():
                pos = (hit.getPosition().X(), hit.getPosition().Y(), hit.getPosition().Z())
                mom = (hit.getMomentum().X(), hit.getMomentum().Y(), hit.getMomentum().Z())

                layer_hit = LayerHit(pos, mom)
                kalman_hits.append(layer_hit)

            particle_hit = Particle(track_id, init_pos, init_mom, actual_hits, kalman_hits)
            particles.append(particle_hit)


# Load layer configuration and sample event row data
layer_df = pd.read_csv(filepath+'layer_radius.csv')

# Load cluster data
clusters_df = pd.read_csv(filepath+'cluster_info.csv')
clusters_df["hit_time"] = clusters_df["gen_time"] + clusters_df["drift_time"]

## Trajectory comparison

In [9]:
# Plot the actual trajectory vs the kalman reconstructed trajectory
def plot_fit(track, side_view, add_wires):
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    kalman_data = [[], [], []]
    actual_data = [[], [], []]

    particle = particles[track]

    for hit in particle.kalman_hits:
        kalman_data[0].append(hit.hit_pos[0])
        kalman_data[1].append(hit.hit_pos[1])
        kalman_data[2].append(hit.hit_pos[2])

    for hit in particle.actual_hits:
        actual_data[0].append(hit.hit_pos[0])
        actual_data[1].append(hit.hit_pos[1])
        actual_data[2].append(hit.hit_pos[2])

    if add_wires:
        for layer_idx in range(len(layer_df)):
            
            # Get the wire layer geometry
            cur_df = layer_df.iloc[layer_idx]
            r = 0.5 * (cur_df['r1'] + cur_df['r2'])
            w_type = cur_df["type"]
            num_wires = cur_df["numWires"]
            wire_length = cur_df["wireLength"]
            delta = 2 * np.pi / num_wires

            # Alternate layers are staggered by half a cell
            offset = 0.0 if cur_df["i"] % 2 == 0 else delta / 2.0
            
            for wire_i in range(num_wires):
                t1 = offset + (wire_i + 0.5) * delta
                if w_type == "stereo+":
                    t2 = offset + (wire_i + 3.5) * delta # Skew 3 wire places forward
                elif w_type == "stereo-":
                    t2 = offset + (wire_i - 2.5) * delta # Skew 3 wire places backward
                else:
                    t2 = t1 # Axial wire so straight across

                # Get the start and end positions of the wire
                x = [r * np.cos(t1), r * np.cos(t2)]
                y = [r * np.sin(t1), r * np.sin(t2)]
                z = [-wire_length, wire_length]

                # Plot the sense wires
                if "stereo" in w_type:
                    ax.plot(x, y, z, color="tab:green", lw=0.5)
                else:
                    ax.plot(x, y, z, color="tab:blue", lw=0.5)

    # Plot fitted and actual tracks
    ax.plot(kalman_data[0], kalman_data[1], kalman_data[2], color="red", label="Kalman fitted")
    ax.plot(actual_data[0], actual_data[1], actual_data[2], color="black", label="Actual")

    # Make plot a 2D view if specified
    if side_view:
        ax.view_init(elev=90, azim=-90)
        
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")

    handles, labels = ax.get_legend_handles_labels()

    # Label the sense wires if added
    if add_wires:
        blue_line = Line2D([0], [0], color='tab:blue', label="Axial wires")
        green_line = Line2D([0], [0], color='tab:green', label="Stereo wires")
        handles.extend([blue_line, green_line])
    
    ax.legend(handles=handles)
    plt.show()

In [50]:
#plot_fit(2, False, False) # Plots 3D view 
#plot_fit(2, True, False)  # Plots 2D view
#plot_fit(2, True, True)  # Plots 2D view with sense wires

## Signal

In [52]:
def plot_signal(m, G, tau):
    # Detector resolution jitter from diffusion values
    quad_sigma = np.std(np.sqrt(clusters_df["long_diffusion"]**2 + clusters_df["trans_diffusion"]**2), ddof=1)
    
    wire_times = []
    gain_list = []
    
    for time in clusters_df["hit_time"]:
        sampled_gain = np.random.gamma(shape=m, scale=G/m)
        gain_list.append(sampled_gain)
        
        quad_ran = random.gauss(0, quad_sigma)
        response_delay = np.random.gamma(shape=2, scale=tau)
        smeared_time = time + response_delay + quad_ran
        wire_times.append(smeared_time)

    # Order the data by increasing time
    sorted_pairs = sorted(zip(wire_times, gain_list))
    wire_times_sorted, gain_sorted = map(list, zip(*sorted_pairs))

    # Make all y values positive
    flipped_gain = abs((np.array(gain_sorted)-G))

    # Insert a value between each value in a list
    def insert_midpoints(lst):
        result = []
        for i in range(len(lst) - 1):
            result.append(lst[i])
            result.append((lst[i] + lst[i+1]) / 2)
        result.append(lst[-1])
        return result

    # Add a value with 0 gain after every point for clarity
    zeroed_times = insert_midpoints(wire_times_sorted[:150])
    zeroed_gains = [x for item in flipped_gain[:150] for x in (item, 0)]

    # Plot the signal function
    plt.figure(figsize=(12, 10))
    plt.plot(zeroed_times[:150], zeroed_gains[:150])
    plt.xlabel("Time [ns]")
    plt.ylabel("Charge/Gain")
    plt.show()

In [64]:
m = 10
G = 20000
tau = 8

#plot_signal(m, G, tau)

## Momentum resolution

In [15]:
def plot_momentum():
    mom_kalman = []
    mom_actual = []

    max_id = 999

    for i in range(max_id):
        cur_lowest = np.inf
        particle = particles[i]

        for hit in particle.kalman_hits:
            mom_mag = np.sqrt(hit.hit_mom[0]**2 + hit.hit_mom[1]**2)
            if mom_mag < cur_lowest: cur_lowest = mom_mag

        if cur_lowest == np.inf: continue

        # Only xy components
        mom_kalman.append(cur_lowest)
        mom_actual.append(particle.init_mom[0]**2 + particle.init_mom[1]**2)

    mom_actual = np.array(mom_actual)
    mom_kalman = np.array(mom_kalman)

    rel_unc = (mom_kalman - mom_actual)/mom_actual

    plt.figure(figsize=(10, 6))
    plt.scatter(mom_actual, mom_kalman)
    plt.xlabel("Actual [MeV]")
    plt.ylabel("Kalman [MeV]")
    plt.show()

    plt.figure(figsize=(10, 6))
    plt.scatter(mom_actual, (mom_kalman - mom_actual))
    plt.xlabel("Actual [MeV]")
    plt.ylabel("Kalman - Actual")
    plt.show()
    
    plt.figure(figsize=(10, 6))
    plt.scatter(mom_actual, rel_unc)
    plt.xlabel("Actual [MeV]")
    plt.ylabel("(Kalman - Actual) / Actual")
    plt.show()

In [17]:
#plot_momentum()